# WM Prediction 2026 – Version 2 (mit Elo Ratings)

Diese Version erweitert das Basismodell um dynamische Elo-Ratings.

Vorteile:
- Gegnerstärke wird berücksichtigt
- Form wird indirekt abgebildet
- Besonders stark bei Nationalmannschaften
- Gute Basis für spätere Wettquoten-Integration


In [81]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import poisson


In [82]:
RESULTS_FILE = "results.csv"

matches = pd.read_csv(RESULTS_FILE)
matches['date'] = pd.to_datetime(matches['date'])
matches = matches.sort_values('date').reset_index(drop=True)
elo_matches = matches.copy()

cutoff = pd.Timestamp.now() - pd.DateOffset(years=20)
matches = matches[matches['date'] >= cutoff].copy()

print(len(matches))


19346


## Elo-System

In [83]:
def get_k_factor(tournament):

    t = str(tournament)

    if t == "FIFA World Cup":
        return 60

    elif "UEFA Euro" in t:
        return 50

    elif "Copa América" in t:
        return 50

    elif "African Cup of Nations" in t:
        return 50

    elif "AFC Asian Cup" in t:
        return 50

    elif "OFC Nations Cup" in t:
        return 50

    elif "Nations League" in t:
        return 40

    elif "qualification" in t.lower():
        return 30

    elif "Friendly" in t:
        return 15

    return 25

In [84]:
INITIAL_ELO = 1500

elo = {}

def get_elo(team):
    return elo.get(team, INITIAL_ELO)

elo_home_before = []
elo_away_before = []
elo_diff_before = []

# Für Elo-Decay
current_year = None

for _, row in matches.iterrows():

    year = row["date"].year

    # ==================================================
    # Elo Decay (5 % Rückkehr Richtung Mittelwert pro Jahr)
    # ==================================================
    if current_year is None:
        current_year = year

    if year > current_year:

        for team in elo:
            elo[team] = (
                INITIAL_ELO
                + (elo[team] - INITIAL_ELO) * 0.98
            )

        current_year = year

    home = row['home_team']
    away = row['away_team']

    eh = get_elo(home)
    ea = get_elo(away)

    elo_home_before.append(eh)
    elo_away_before.append(ea)
    elo_diff_before.append(eh - ea)

    expected_home = 1 / (1 + 10 ** ((ea - eh) / 400))
    expected_away = 1 - expected_home

    if row['home_score'] > row['away_score']:
        actual_home = 1
        actual_away = 0

    elif row['home_score'] < row['away_score']:
        actual_home = 0
        actual_away = 1

    else:
        actual_home = 0.5
        actual_away = 0.5

    # Turnierabhängiger K-Faktor
    k = get_k_factor(row["tournament"])

    # ==================================================
    # Torabstand berücksichtigen
    # ==================================================
    goal_diff = min(
        abs(row["home_score"] - row["away_score"]),
        4
    )


    elo[home] = (
        eh
        + k
        * (actual_home - expected_home)
    )

    elo[away] = (
        ea
        + k
        * (actual_away - expected_away)
    )

matches['elo_home'] = elo_home_before
matches['elo_away'] = elo_away_before
matches['elo_diff'] = elo_diff_before

## Turnier- und Zeitgewichtung

In [85]:
def tournament_weight(tournament):

    t = str(tournament)

    if 'FIFA World Cup' in t:
        return 20

    if 'UEFA Euro' in t:
        return 15

    if 'Copa América' in t:
        return 15

    if 'African Cup of Nations' in t:
        return 15

    if 'AFC Asian Cup' in t:
        return 15

    if 'OFC Nations Cup' in t:
        return 15

    if 'Nations League' in t:
        return 8

    if 'qualification' in t.lower():
        return 8

    if 'Friendly' in t:
        return 1

    return 2

matches['tournament_weight'] = matches['tournament'].apply(tournament_weight)

days_old = (pd.Timestamp.now() - matches['date']).dt.days

matches['time_weight'] = np.exp(-days_old / 1460)

matches['weight'] = (
    matches['tournament_weight']
    * matches['time_weight']
)


## Modell-Datensatz

In [86]:
home = matches[[
    'home_team',
    'away_team',
    'home_score',
    'weight',
    'elo_diff'
]].copy()

home.columns = [
    'team',
    'opponent',
    'goals',
    'weight',
    'elo_diff'
]

away = matches[[
    'away_team',
    'home_team',
    'away_score',
    'weight',
    'elo_diff'
]].copy()

away.columns = [
    'team',
    'opponent',
    'goals',
    'weight',
    'elo_diff'
]

away['elo_diff'] = -away['elo_diff']

goal_model_data = pd.concat([home, away], ignore_index=True)


## Poisson + Elo Modell

In [87]:
model = smf.glm(
    formula='goals ~ team + opponent + elo_diff',
    data=goal_model_data,
    family=sm.families.Poisson(),
    freq_weights=goal_model_data['weight']
).fit()

print(model.summary())


                 Generalized Linear Model Regression Results                  
Dep. Variable:                  goals   No. Observations:                38548
Model:                            GLM   Df Residuals:                 76069.90
Model Family:                 Poisson   Df Model:                          632
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.0378e+05
Date:                Thu, 04 Jun 2026   Deviance:                       80601.
Time:                        19:33:11   Pearson chi2:                 7.27e+04
No. Iterations:                    21   Pseudo R-squ. (CS):             0.6967
Covariance Type:            nonrobust                                         
                                                   coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------

## Aktuelle Elo-Tabelle

In [88]:
elo_table = (
    pd.DataFrame(
        list(elo.items()),
        columns=['team','elo']
    )
    .sort_values('elo', ascending=False)
)

elo_table.head(25)


,team,elo
2,Spain,1899.125597
13,Argentina,1873.066977
4,France,1850.906696
79,Morocco,1836.909855
15,England,1803.492670
20,Portugal,1802.891128
75,Senegal,1795.835689
29,Japan,1793.563097
30,Brazil,1787.420732
109,Colombia,1785.419548


## Vorhersagefunktion

In [90]:
def predict_goals(team, opponent):

    elo_diff = get_elo(team) - get_elo(opponent)

    df = pd.DataFrame({
        'team':[team],
        'opponent':[opponent],
        'elo_diff':[elo_diff]
    })

    return float(model.predict(df)[0])


In [91]:
def top_results(team1, team2, max_goals=8):

    g1 = predict_goals(team1, team2)
    g2 = predict_goals(team2, team1)

    matrix = np.outer(
        poisson.pmf(range(max_goals + 1), g1),
        poisson.pmf(range(max_goals + 1), g2)
    )

    rows = []

    for i in range(max_goals + 1):
        for j in range(max_goals + 1):
            rows.append((i,j,matrix[i,j]))

    rows.sort(key=lambda x: x[2], reverse=True)

    return pd.DataFrame(
        rows[:10],
        columns=['team1_goals','team2_goals','probability']
    )


In [92]:
# Beispiel

top_results('France','Germany')


,team1_goals,team2_goals,probability
0,1,1,0.120913
1,1,0,0.092769
2,2,1,0.086841
3,0,1,0.084177
4,1,2,0.078798
5,2,0,0.066628
6,0,0,0.064584
7,2,2,0.056593
8,0,2,0.054857
9,3,1,0.041580


# Version 3

Nächste Ausbaustufe:

- Elo mit Turniergewichtung
- Elo mit Torabstand
- Dixon-Coles Modell
- Form der letzten 5 Spiele
- Automatische WM-Teilnehmer
- Wettquoten-Features
